# 52 — Weather Validation and Fallback Audit
Validate weather outputs before they are used as evidence.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def normalise_date_col(df):
    for c in df.columns:
        if c.lower() in {"date","day","datetime","timestamp","time","time_utc","publishedat","published_at"}:
            out = df.copy()
            out["date"] = pd.to_datetime(out[c], errors="coerce").dt.normalize()
            return out, c
    return df.copy(), None

def list_output_files():
    rows = []
    if not OUTPUTS.exists():
        return pd.DataFrame(columns=["path","size_bytes","sha256"])
    for p in sorted(OUTPUTS.rglob("*")):
        if p.is_file():
            rows.append({
                "path": str(p.relative_to(PROJECT_ROOT)),
                "size_bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

run_cfg = load_yaml(CONFIGS / "run.yml")
phase_cfg = load_yaml(CONFIGS / "phase50.yml")
sites_df = pd.DataFrame(run_cfg.get("sites", []))
phase_sites = pd.DataFrame(phase_cfg.get("additional_sites", []))
if not phase_sites.empty:
    existing = set(sites_df.get("id", pd.Series(dtype=str)).astype(str).tolist()) if not sites_df.empty else set()
    phase_sites = phase_sites[~phase_sites["id"].astype(str).isin(existing)]
    sites_df = pd.concat([sites_df, phase_sites], ignore_index=True)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])
date_from = run_cfg.get("run", {}).get("date_from", "2025-01-01")
date_to = run_cfg.get("run", {}).get("date_to", "2025-01-31")

In [ ]:
step = "52_weather_validation_and_fallback_audit"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

thresholds = load_yaml(CONFIGS / "evidence_thresholds.yml")
min_rows = int(thresholds.get("weather", {}).get("min_hourly_rows", 24))
weather_candidates = [
    OUTPUTS / "05_metoffice_datahub_weather" / "weather_hourly.csv",
    OUTPUTS / "32_newhaven_weather_ground_news_fusion" / "weather_hourly.csv",
    OUTPUTS / "newhaven_fusion" / "weather_hourly.csv",
]

audit_rows, chosen, weather_df = [], None, pd.DataFrame()
for p in weather_candidates:
    if p.exists():
        df, meta = safe_read_csv(p)
        if not df.empty:
            df, dc = normalise_date_col(df)
            audit_rows.append({
                "path": str(p.relative_to(PROJECT_ROOT)),
                "rows": int(len(df)),
                "columns": int(df.shape[1]),
                "sha256": meta.get("sha256", ""),
            })
            if chosen is None:
                chosen = p
                weather_df = df.copy()

audit = pd.DataFrame(audit_rows)
verdict = {
    "chosen_weather_path": str(chosen.relative_to(PROJECT_ROOT)) if chosen else "",
    "weather_ready": bool(chosen is not None and len(weather_df) >= min_rows),
    "candidate_count": int(len(audit_rows)),
}
mof_manifest = OUTPUTS / "05_metoffice_datahub_weather" / "manifest.json"
if mof_manifest.exists():
    text = json.dumps(load_json(mof_manifest))
    verdict["metoffice_401_detected"] = "401" in text
else:
    verdict["metoffice_401_detected"] = False

audit.to_csv(out / "weather_audit.csv", index=False)
if chosen is not None and not weather_df.empty:
    weather_df.to_csv(out / "weather_validated_hourly.csv", index=False)
write_json(out / "weather_validation_summary.json", verdict)

manifest = manifest_base(step, [CONFIGS / "evidence_thresholds.yml"])
for p in [out / "weather_audit.csv", out / "weather_validation_summary.json"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
if (out / "weather_validated_hourly.csv").exists():
    manifest["artifacts"].append({"path": str(out / "weather_validated_hourly.csv"), "sha256": sha256_file(out / "weather_validated_hourly.csv")})
write_json(out / "manifest.json", manifest)
display(audit)
print(verdict)